# Lesson 01 - Johdanto tekoälyagentteihin

Tervetuloa **AI Agents for Beginners** -kurssin ensimmäiseen oppituntiin!

**Tekoälyagentti** on ohjelma, joka käyttää suurta kielimallia (LLM) päättelymoottorinaan ja voi tehdä *toimia* reaalimaailmassa — kutsua rajapintoja, kysellä tietokantoja tai suorittaa koodia — saavuttaakseen tavoitteen käyttäjän puolesta.

Tässä muistikirjassa rakennat ensimmäisen agenttisi: **Matka-agentin**, joka suosittelee lomakohteita. Matkan varrella opit, miten:

1. Yhdistää Azure AI Foundry Agent Serviceen käyttäen **Microsoft Agent Frameworkia**.
2. Antaa agentille **työkalu** — tavallinen Python-funktio, jota se voi kutsua.
3. Suorittaa agentti ja tarkastella sen vastausta.
4. Suodattaa agentin vastaus token kerrallaan.


## Määritys

Ennen tämän muistikirjan suorittamista varmista, että sinulla on:

1. **Azure AI Foundry -projekti**, jossa on otettu käyttöön chat-malli (esim. `gpt-4o-mini`).
2. **Kirjautunut Azure CLI:llä** — suorita `az login` terminaalissasi.
3. **Asetettu vaaditut ympäristömuuttujat:**
   - `AZURE_AI_PROJECT_ENDPOINT` — Azure AI Foundry -projektisi päätepiste.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — käyttöön otetun mallin nimi.

Alla oleva solu asentaa tarvitsemasi Python-paketit.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ensimmäisen agenttisi luominen

Agentilla on kaksi tarvetta:

- **Ohjeet**, jotka kertovat sille *kuka se on* ja *miten käyttäytyä* (järjestelmän kehotus).
- **Työkalut** — Python-funktiot, jotka on koristeltu `@tool`-merkinnällä, ja joita agentti voi kutsua tiedon hakemiseen tai toimien suorittamiseen.

Alla määrittelemme yksinkertaisen työkalun, joka palauttaa listan suosituista lomakohteista. Agentti käyttää tätä työkalua, kun käyttäjä pyytää matkasuosituksia.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Suoratoistovastaukset

Vuorovaikutteisemman käyttökokemuksen saamiseksi voit **suoratoistaa** agentin vastauksen. Täyden vastauksen odottamisen sijaan agentti lähettää tekstin osissa sitä mukaa kun ne valmistuvat. Tämä on erityisen hyödyllistä chat-käyttöliittymissä, joissa haluat näyttää tulosteen reaaliajassa.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Yhteenveto

Tässä oppitunnissa opit miten:

- **Luoda tarjoaja**, joka yhdistää Azure AI Foundry Agent Serviceen `AzureAIProjectAgentProvider`-luokan kautta.
- **Määritellä työkalu** käyttämällä `@tool`-koristetta, jotta agentti voi kutsua Python-funktioitasi.
- **Suorittaa agentti** käyttäjän viestillä ja tulostaa sen vastaus.
- **Virtaa vastauksia** reaaliaikaista tulostusta varten.

Seuraavassa oppitunnissa tutkimme agenttipohjaisia kehyksiä tarkemmin ja opimme, miten agentteja voidaan varustaa tehokkaammilla työkaluilla ja monivaiheisella päättelykyvyllä.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:  
Tämä asiakirja on käännetty tekoälypohjaisella käännöspalvelulla [Co-op Translator](https://github.com/Azure/co-op-translator). Pyrimme tarkkuuteen, mutta ota huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja alkuperäisellä kielellä on virallinen lähde. Tärkeissä tiedoissa suosittelemme ammattilaisen tekemää käännöstä. Emme vastaa tämän käännöksen käytöstä mahdollisesti aiheutuvista väärinkäsityksistä tai virhetulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
